In [0]:
# ============================================================
# 1. LOAD SILVER TABLE
# ============================================================

from pyspark.sql import functions as F

SILVER_TABLE = "streaming_sentiment_intelligence.default.silver_reviews"

df_silver = spark.table(SILVER_TABLE)

print("Silver records:", df_silver.count())
print("Silver columns:", len(df_silver.columns))

display(df_silver.limit(10))

Silver records: 707
Silver columns: 18


review_text,sentiment,timestamp,user,platform,hashtags,retweets,likes,country,year,month,day,hour,_ingestion_timestamp,text_length,word_count,char_count_no_space,engagement_score
Excited about the upcoming weekend getaway!,Positive,2023-01-15T18:20:00.000Z,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18,2026-08-19T13:55:38.611Z,43,6,38,23.0
"New year, new fitness goals! 💪",Positive,2023-01-18T18:00:00.000Z,FitJourney,Instagram,#NewYear #FitnessGoals,28.0,55.0,USA,2023,1,18,18,2026-08-19T13:55:38.611Z,30,6,25,83.0
A cozy evening with a good movie.,Positive,2023-01-29T20:20:00.000Z,MovieNight,Twitter,#CozyNight #MovieTime,18.0,35.0,Canada,2023,1,29,20,2026-08-19T13:55:38.611Z,33,7,27,53.0
Overflowing happiness: welcoming a new family member!,Happiness,2023-02-22T10:00:00.000Z,NewParentJoy,Instagram,#Happiness #NewFamilyMember,30.0,60.0,UK,2023,2,22,10,2026-08-19T13:55:38.611Z,53,7,47,90.0
Laughter is the key to joy—attending a stand-up comedy show.,Joy,2023-02-22T19:30:00.000Z,StandUpFan,Facebook,#Joy #StandUpComedy,22.0,45.0,Canada,2023,2,22,19,2026-08-19T13:55:38.611Z,60,10,51,67.0
Reflecting on the beauty of diversity in our world.,Acceptance,2023-03-01T10:45:00.000Z,DiversityLover,Twitter,#Acceptance #Diversity,22.0,45.0,Australia,2023,3,1,10,2026-08-19T13:55:38.611Z,51,9,43,67.0
Elation after achieving a personal goal.,Elation,2021-03-02T09:30:00.000Z,GoalAchiever2,Facebook,#Elation #PersonalAchievement,25.0,50.0,USA,2021,3,2,9,2026-08-19T13:55:38.611Z,40,6,35,75.0
Enthusiasm for a DIY home improvement project.,Enthusiasm,2018-04-15T16:00:00.000Z,DIYEnthusiast,Facebook,#Enthusiasm #HomeImprovement,15.0,30.0,Australia,2018,4,15,16,2026-08-19T13:55:38.611Z,46,7,40,45.0
Elation after a surprise reunion with a childhood friend.,Elation,2021-02-05T09:30:00.000Z,ChildhoodJoy,Facebook,#Elation #FriendshipReunion,25.0,50.0,Canada,2021,2,5,9,2026-08-19T13:55:38.611Z,57,9,49,75.0
"Floating through the day with an air of indifference, detached from the mundane happenings around.",Indifference,2023-03-22T16:30:00.000Z,AloofObserver,Facebook,#Indifference #FloatingThroughLife,10.0,20.0,Canada,2023,3,22,16,2026-08-19T13:55:38.611Z,98,15,84,30.0


In [0]:
# ============================================================
# 2. GOLD - SENTIMENT SUMMARY
# ============================================================

gold_sentiment = (
    df_silver
    .groupBy("sentiment")
    .agg(
        F.count("*").alias("total_reviews"),
        F.round(F.avg("likes"), 2).alias("avg_likes"),
        F.round(F.avg("retweets"), 2).alias("avg_retweets"),
        F.round(F.avg("engagement_score"), 2).alias("avg_engagement"),
        F.round(F.avg("text_length"), 2).alias("avg_text_length"),
        F.round(F.avg("word_count"), 2).alias("avg_word_count")
    )
    .orderBy(F.desc("total_reviews"))
)

display(gold_sentiment)

sentiment,total_reviews,avg_likes,avg_retweets,avg_engagement,avg_text_length,avg_word_count
Positive,45,37.78,18.87,56.64,40.13,6.58
Joy,44,49.27,24.7,73.98,107.39,15.59
Excitement,37,50.27,25.49,75.76,92.35,14.49
Neutral,18,40.5,20.56,61.06,84.72,12.33
Contentment,18,52.0,26.11,78.11,98.39,14.83
Gratitude,17,49.41,24.71,74.12,92.65,13.88
Curiosity,16,43.0,21.5,64.5,100.06,14.94
Serenity,15,42.4,21.13,63.53,80.47,13.8
Happy,14,45.14,23.07,68.21,98.29,14.0
Despair,11,35.64,17.91,53.55,82.45,13.55


In [0]:
GOLD_SENTIMENT_TABLE = "streaming_sentiment_intelligence.default.gold_sentiment_summary"

(
    gold_sentiment.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_SENTIMENT_TABLE)
)

print("✅ gold_sentiment_summary created")

✅ gold_sentiment_summary created


In [0]:
# ============================================================
# 3. GOLD - PLATFORM ANALYSIS
# ============================================================

gold_platform = (
    df_silver
    .groupBy("platform", "sentiment")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("likes"), 2).alias("avg_likes"),
        F.round(F.avg("retweets"), 2).alias("avg_retweets"),
        F.round(F.avg("engagement_score"), 2).alias("avg_engagement")
    )
    .orderBy(F.desc("review_count"))
)

display(gold_platform)

platform,sentiment,review_count,avg_likes,avg_retweets,avg_engagement
Facebook,Joy,20,46.8,23.5,70.3
Instagram,Positive,16,40.31,20.19,60.5
Facebook,Positive,16,34.38,17.06,51.44
Instagram,Joy,14,56.5,28.36,84.86
Instagram,Excitement,13,51.31,26.15,77.46
Twitter,Excitement,13,49.85,25.31,75.15
Twitter,Positive,13,38.85,19.46,58.31
Facebook,Excitement,11,49.55,24.91,74.45
Twitter,Joy,10,44.1,22.0,66.1
Twitter,Gratitude,7,50.0,25.0,75.0


In [0]:
GOLD_PLATFORM_TABLE = "streaming_sentiment_intelligence.default.gold_platform_analysis"

(
    gold_platform.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_PLATFORM_TABLE)
)

print("✅ gold_platform_analysis created")

✅ gold_platform_analysis created


In [0]:
# ============================================================
# 4. GOLD - COUNTRY ANALYSIS
# ============================================================

gold_country = (
    df_silver
    .groupBy("country", "sentiment")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("likes"), 2).alias("avg_likes"),
        F.round(F.avg("retweets"), 2).alias("avg_retweets"),
        F.round(F.avg("engagement_score"), 2).alias("avg_engagement")
    )
    .orderBy(F.desc("review_count"))
)

display(gold_country)

country,sentiment,review_count,avg_likes,avg_retweets,avg_engagement
USA,Positive,18,43.89,22.0,65.89
USA,Joy,17,46.35,23.18,69.53
Canada,Joy,15,50.47,25.4,75.87
UK,Excitement,11,49.18,25.36,74.55
Canada,Excitement,11,49.55,25.18,74.73
UK,Positive,10,32.5,16.2,48.7
USA,Excitement,10,49.9,25.0,74.9
UK,Joy,9,48.11,24.11,72.22
Canada,Positive,8,27.5,13.63,41.13
India,Positive,7,43.57,21.71,65.29


In [0]:
GOLD_COUNTRY_TABLE = "streaming_sentiment_intelligence.default.gold_country_analysis"

(
    gold_country.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_COUNTRY_TABLE)
)

print("✅ gold_country_analysis created")

✅ gold_country_analysis created


In [0]:
# ============================================================
# 5. GOLD - TIME ANALYSIS
# ============================================================

gold_time = (
    df_silver
    .groupBy(
        "year",
        "month",
        "day",
        "hour",
        "sentiment"
    )
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("likes"), 2).alias("avg_likes"),
        F.round(F.avg("retweets"), 2).alias("avg_retweets"),
        F.round(F.avg("engagement_score"), 2).alias("avg_engagement")
    )
    .orderBy(
        "year",
        "month",
        "day",
        "hour"
    )
)

display(gold_time)

GOLD_TIME_TABLE = "streaming_sentiment_intelligence.default.gold_time_analysis"

(
    gold_time.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TIME_TABLE)
)

print("✅ gold_time_analysis created")

year,month,day,hour,sentiment,review_count,avg_likes,avg_retweets,avg_engagement
2010,5,15,15,Elation,1,40.0,20.0,60.0
2010,8,15,10,Contentment,1,60.0,30.0,90.0
2010,11,12,20,Contentment,1,60.0,30.0,90.0
2011,6,20,14,Contentment,1,50.0,25.0,75.0
2011,7,22,18,Serenity,1,45.0,22.0,67.0
2011,8,28,18,Fulfillment,1,45.0,22.0,67.0
2011,9,22,19,Serenity,1,45.0,22.0,67.0
2012,2,18,14,Elation,1,40.0,20.0,60.0
2012,3,10,8,Gratitude,1,30.0,15.0,45.0
2012,3,30,11,Gratitude,1,30.0,15.0,45.0


✅ gold_time_analysis created


In [0]:
# ============================================================
# 6. VERIFY GOLD TABLES
# ============================================================

tables = [
    "gold_sentiment_summary",
    "gold_platform_analysis",
    "gold_country_analysis",
    "gold_time_analysis"
]

for table in tables:
    full_table = f"streaming_sentiment_intelligence.default.{table}"
    
    count = spark.table(full_table).count()
    
    print(f"✅ {table}: {count} rows")

✅ gold_sentiment_summary: 191 rows
✅ gold_platform_analysis: 326 rows
✅ gold_country_analysis: 400 rows
✅ gold_time_analysis: 696 rows
